If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec35_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 35: Classification
v.ekc

A new kind of prediction: not a number, but a **category** — sick or well, real or counterfeit. Today: what classifiers are, and the simplest one that works — *find the most similar example you've seen, and copy its label*. (Slides: KL Lecture 35 deck.)

**Today's sections:**
1. Classifying patients
2. Classifying banknotes
3. The nearest neighbor classifier
4. Decision boundaries

In [ ]:
from datascience import *
import numpy as np
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

> **Setup cells:** `standard_units`, `distance`, and the plotting/classifier helpers used below. Run and move on.

In [ ]:
def standard_units(x):
    return (x - np.mean(x))/np.std(x)

In [ ]:
def distance(point1, point2):
    """The distance between two arrays of numbers."""
    return np.sqrt(np.sum((point1 - point2)**2))

def all_distances(training, point):
    """The distance between p (an array of numbers) and the numbers in row i of attribute_table."""
    attributes = training.drop('Class')
    def distance_from_point(row):
        return distance(point, np.array(row))
    return attributes.apply(distance_from_point)

def table_with_distances(training, point):
    """A copy of the training table with the distance from each row to array p."""
    return training.with_column('Distance', all_distances(training, point))

def closest(training, point, k):
    """A table containing the k closest rows in the training table to array p."""
    with_dists = table_with_distances(training, point)
    sorted_by_distance = with_dists.sort('Distance')
    topk = sorted_by_distance.take(np.arange(k))
    return topk

def majority(topkclasses):
    """1 if the majority of the "Class" column is 1s, and 0 otherwise."""
    ones = topkclasses.where('Class', are.equal_to(1)).num_rows
    zeros = topkclasses.where('Class', are.equal_to(0)).num_rows
    if ones > zeros:
        return 1
    else:
        return 0

def classify(training, p, k):
    """Classify an example with attributes p using k-nearest neighbor classification with the given training table."""
    closestk = closest(training, p, k)
    topkclasses = closestk.select('Class')
    return majority(topkclasses)

def show_closest(point):
    """point = array([x,y]) 
    gives the coordinates of a new point
    shown in red"""
    
    HemoGl = ckd.drop('White Blood Cell Count', 'Color')
    t = closest(HemoGl, point, 1)
    x_closest = t.row(0).item(1)
    y_closest = t.row(0).item(2)
    ckd.scatter('Hemoglobin', 'Glucose', group='Color')
    plt.scatter(point.item(0), point.item(1), color='red', s=30)
    plt.plot(make_array(point.item(0), x_closest), make_array(point.item(1), y_closest), color='k', lw=2);

In [ ]:
def plot_all_points(test_grid):
    test_grid.scatter('Hemoglobin', 'Glucose', color='red', alpha=0.4, s=30)

    plt.scatter(ckd.column('Hemoglobin'), ckd.column('Glucose'), c=ckd.column('Color'), edgecolor='k')

    plt.xlim(-2, 2)
    plt.ylim(-2, 2);
    
def classify_grid(training, test, k):
    c = make_array()
    for i in range(test.num_rows):
        # Run the classifier on the ith patient in the test set
        c = np.append(c, classify(training, make_array(test.row(i)), k))   
    return c

def plot_all_points_classified(test_grid):
    c = classify_grid(ckd.drop('White Blood Cell Count', 'Color'), test_grid, 1)
    test_grid = test_grid.with_column('Class', c).join('Class', color_table)
    test_grid.scatter('Hemoglobin', 'Glucose', group='Color', alpha=0.4, s=30)

    plt.scatter(ckd.column('Hemoglobin'), ckd.column('Glucose'), c=ckd.column('Color'), edgecolor='k')

    plt.xlim(-2, 2)
    plt.ylim(-2, 2);

---
## 1. Classifying Patients 🩺

Chronic kidney disease data: for each patient, lab measurements and a `Class` (1 = has CKD). The goal: predict the class of a **new** patient from their measurements alone. (KL deck, slides 17–19.)

In [ ]:
ckd = Table.read_table('ckd.csv').relabeled('Blood Glucose Random', 'Glucose')
ckd.show(3)

In [ ]:
ckd.group('Class')

In [ ]:
ckd.scatter('White Blood Cell Count', 'Glucose', group='Class')

In [ ]:
ckd.scatter('Hemoglobin', 'Glucose', group='Class')

The two groups separate so cleanly that we can classify with hand-written thresholds:

In [ ]:
# we want to be able to way to predict the class of someone
# without having to plot & eye ball this graph every time.
#
# one way to do this is to put some thresholds into code

max_glucose_for_0 = ckd.where('Class',are.equal_to(0)).column('Glucose').max()
min_hemoglobin_for_0 = ckd.where('Class',are.equal_to(0)).column('Hemoglobin').min()

In [ ]:
def classify_manually(hemoglobin, glucose):
    if hemoglobin > min_hemoglobin_for_0 or glucose < max_glucose_for_0:
        return 0
    else:
        return 1

In [ ]:
# Let's try our classifier!
classify_manually(15, 100)

In [ ]:
classify_manually(10, 300)

> **That worked here because the classes barely overlap.** Real data is messier — we need a method that doesn't require a human squinting at a scatter plot. But first, see how much messier it can get:

---
## 2. Classifying Banknotes 💵

Genuine vs. counterfeit banknotes, measured by image statistics. Some attribute pairs separate the classes well, others poorly — and sometimes you need a third dimension.

In [ ]:
banknotes = Table.read_table('banknote.csv')
banknotes

In [ ]:
banknotes.group('Class')

In [ ]:
banknotes.scatter('WaveletVar', 'WaveletCurt', group='Class')

In [ ]:
banknotes.scatter('WaveletSkew', 'Entropy', group='Class')

In [ ]:
fig = plt.figure(figsize = (10, 7))
ax = plt.axes(projection ="3d")
 
# Creating plot
ax.scatter3D(banknotes.column('WaveletSkew'), 
             banknotes.column('WaveletVar'),
             banknotes.column('WaveletCurt'), c = banknotes.column('Class'),
             cmap='viridis',
             s=50);

> **Attributes are your choice** — and picking ones that *separate the classes* matters more than the fancy classifier. (KL deck, slides 24–25.)

---
## 3. The Nearest Neighbor Classifier

The plan: convert attributes to **standard units** (so no attribute dominates just by having bigger numbers), then classify a new point by **copying the class of its closest neighbor**.

### 📋 Board Reference

| Step | Code |
|---|---|
| distance between points | `np.sqrt(np.sum((point1 - point2)**2))` |
| fair scale | convert each attribute to standard units first |
| nearest neighbor | training row with the smallest distance |
| prediction | that neighbor's class |

In [ ]:
ckd

In [ ]:
# convert features into standard units
ckd = Table().with_columns(
    'Hemoglobin', standard_units(ckd.column('Hemoglobin')),
    'Glucose', standard_units(ckd.column('Glucose')),
    'White Blood Cell Count', standard_units(ckd.column('White Blood Cell Count')),
    'Class', ckd.column('Class')
)
ckd

In [ ]:
color_table = Table().with_columns(
    'Class', make_array(0, 1),
    'Color', make_array('darkblue', 'gold')
)
ckd = ckd.join('Class', color_table)

In [ ]:
ckd.scatter('Hemoglobin', 'Glucose', group='Color')

In [ ]:
# In this example, Alice's Hemoglobin is 0 and her Glucose is 1.5.
alice = make_array(0, 1.5)
show_closest(alice)

---
## 4. Decision Boundaries

Alice at (0, 1.5) is classified gold. Move her a little — (0, 0.95) — and the answer flips: she's crossed the **decision boundary**, the invisible line where the nearest neighbor changes.

In [ ]:
alice = make_array(0, 0.95)
show_closest(alice)

In [ ]:
# Create a grid of all points
x_array = make_array()
y_array = make_array()
for x in np.arange(-2, 2.1, 0.1):
    for y in np.arange(-2, 2.1, 0.1):
        x_array = np.append(x_array, x)
        y_array = np.append(y_array, y)

test_grid = Table().with_columns(
    'Hemoglobin', x_array,
    'Glucose', y_array
)

In [ ]:
plot_all_points(test_grid)

In [ ]:
plot_all_points_classified(test_grid)

### Try It 1: Who's nearest? 😊

1. Use the `distance` function to compute how far Alice at `make_array(0, 1.5)` is from a patient at `make_array(-0.4, 1.1)`, and from one at `make_array(1.1, 2.0)`. Which is her nearest neighbor of the two?

In [ ]:
# 1. two distances from alice


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Straight-line (Euclidean) distance, exactly like geometry class.*

```python
# 1.
alice = make_array(0, 1.5)
distance(alice, make_array(-0.4, 1.1))   # ≈ 0.57  ← nearer
distance(alice, make_array(1.1, 2.0))    # ≈ 1.21
```

</details>

> **One more thing — bias** (KL deck, slide 33): a classifier learns from its training data. If the training set under-represents a group, predictions for that group will be worse. "The algorithm decided" is never the end of the ethical story.

---
> **Today's takeaway:** classification = predicting categories; nearest-neighbor does it with nothing but a distance function and honest training data. Next: letting *k* neighbors vote.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — distance
```python
def distance(point1, point2):
    return np.sqrt(np.sum((point1 - point2)**2))
```